In [ ]:

import csv
import re
import time
from dataclasses import dataclass, asdict
from datetime import datetime, date
from typing import Optional

import requests
from bs4 import BeautifulSoup
from dateutil import parser as dtparser


TARGET_CHANNELS = [
    "postypashki_old"
]

# Окна дат вокруг всплесков из sales_mart (по UTC/локальному времени поста)
DATE_WINDOWS = [
    (date(2026, 8, 8), date(2026, 8, 10)),   # основной всплеск
    (date(2026, 9, 4), date(2026, 9, 6)),    # основной всплеск
    (date(2026, 8, 22), date(2026, 8, 23)),  # дополнительное окно
]

OUTPUT_CSV = "telegram_marketing_history.csv"

# сколько страниц назад листать максимум (защита от бесконечного цикла) —
# одна страница t.me/s/ отдаёт ~20 постов
MAX_PAGES_PER_CHANNEL = 400

# пауза между запросами
REQUEST_DELAY_SEC = 0.6

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124 Safari/537.36"
}


# Классификация эвристики "по словам"
KEYWORDS = {
    "discount": [
        r"скидк", r"промокод", r"со скидкой", r"\-\s?\d{1,2}\s?%",
        r"спецпредложен", r"дешевле", r"выгодн",
    ],
    "launch": [
        r"открываем набор", r"старт(ует)? курс", r"новый поток",
        r"старт продаж", r"начинаем набор", r"запуск(аем)?",
        r"набор на курс", r"набор на \d",
    ],
    "sale_post": [
        r"цена\s?\d", r"стоимост", r"₽", r"руб\.", r"купить", r"записаться",
        r"для записи", r"оплат", r"успей",
    ],
    "native_integration": [
        r"реклама", r"партнер", r"партнёр", r"по промокоду",
    ],
}
COMPILED_KEYWORDS = {
    tag: [re.compile(p, re.IGNORECASE) for p in patterns]
    for tag, patterns in KEYWORDS.items()
}


def classify_text(text: str, is_forwarded: bool) -> list:
    tags = []
    for tag, patterns in COMPILED_KEYWORDS.items():
        if any(p.search(text) for p in patterns):
            tags.append(tag)
    if is_forwarded:
        tags.append("forwarded_possible_integration")
    return tags or ["other"]



@dataclass
class Post:
    channel: str
    msg_id: Optional[int]
    url: str
    datetime_utc: Optional[str]
    text: str
    views: Optional[str]
    reactions_total: Optional[int]
    is_forwarded: bool
    forwarded_from: Optional[str]
    has_media: bool
    links_in_text: str
    tags: str


#парсинг

def fetch_page(channel: str, before_id: Optional[int] = None) -> Optional[str]:
    url = f"https://t.me/s/{channel}"
    params = {"before": before_id} if before_id else {}
    try:
        resp = requests.get(url, params=params, headers=HEADERS, timeout=15)
    except requests.RequestException as e:
        print(f"  [!] ошибка запроса: {e}")
        return None
    if resp.status_code != 200:
        print(f"  [!] статус {resp.status_code} для {channel}, before={before_id}")
        return None
    return resp.text


def parse_message_div(div, channel: str) -> Optional[Post]:
    # id поста / ссылка
    link_tag = div.get("data-post")  # формат "channel/12345"
    msg_id = None
    url = ""
    if link_tag:
        url = f"https://t.me/{link_tag}"
        try:
            msg_id = int(link_tag.split("/")[-1])
        except ValueError:
            pass

    # дата
    time_tag = div.select_one("time.time")
    dt_iso = None
    if time_tag and time_tag.get("datetime"):
        try:
            dt_iso = dtparser.isoparse(time_tag["datetime"]).isoformat()
        except (ValueError, TypeError):
            pass

    # текст
    text_tag = div.select_one(".tgme_widget_message_text")
    text = text_tag.get_text(separator="\n").strip() if text_tag else ""

    # просмотр
    views_tag = div.select_one(".tgme_widget_message_views")
    views = views_tag.get_text(strip=True) if views_tag else None

    # реакции (сумма)
    reaction_spans = div.select(".tgme_widget_message_reaction_emoji + .tgme_widget_message_reaction_count") \
        if div.select(".tgme_widget_message_reaction_count") else []
    reaction_counts = [c.get_text(strip=True) for c in div.select(".tgme_widget_message_reaction_count")]
    reactions_total = None
    if reaction_counts:
        try:
            reactions_total = sum(int(re.sub(r"[^\d]", "", c) or 0) for c in reaction_counts)
        except ValueError:
            reactions_total = None

    # форвард (частый признак нативной интеграции / репоста рекламы)
    fwd_tag = div.select_one(".tgme_widget_message_forwarded_from_name")
    is_forwarded = fwd_tag is not None
    forwarded_from = fwd_tag.get_text(strip=True) if fwd_tag else None

    # медиа
    has_media = bool(div.select_one(".tgme_widget_message_photo, .tgme_widget_message_video, .tgme_widget_message_document"))

    # ссылки в тексте
    links = [a["href"] for a in (text_tag.select("a") if text_tag else []) if a.get("href")]

    if not text and not has_media:
        return None  # служебный блок без контента

    tags = classify_text(text, is_forwarded)

    return Post(
        channel=channel,
        msg_id=msg_id,
        url=url,
        datetime_utc=dt_iso,
        text=text.replace("\n", " | ")[:2000],
        views=views,
        reactions_total=reactions_total,
        is_forwarded=is_forwarded,
        forwarded_from=forwarded_from,
        has_media=has_media,
        links_in_text=" | ".join(links[:10]),
        tags=";".join(tags),
    )


def in_any_window(post_date: date) -> bool:
    return any(start <= post_date <= end for start, end in DATE_WINDOWS)


def earliest_window_start() -> date:
    return min(start for start, _ in DATE_WINDOWS)


def collect_channel(channel: str) -> list:
    print(f"\n=== Канал: {channel} ===")
    all_posts = []
    before_id = None
    stop = False

    for page_num in range(MAX_PAGES_PER_CHANNEL):
        html = fetch_page(channel, before_id)
        if not html:
            break

        soup = BeautifulSoup(html, "html.parser")
        message_divs = soup.select(".tgme_widget_message")
        if not message_divs:
            print("  постов на странице не найдено, останавливаюсь")
            break

        page_min_id = None
        for div in message_divs:
            post = parse_message_div(div, channel)
            if post is None:
                continue
            if post.msg_id is not None:
                page_min_id = post.msg_id if page_min_id is None else min(page_min_id, post.msg_id)
            all_posts.append(post)

            if post.datetime_utc:
                post_date = dtparser.isoparse(post.datetime_utc).date()
                # если ушли раньше самого раннего окна — можно прекращать
                if post_date < earliest_window_start():
                    stop = True

        print(f"  страница {page_num + 1}: +{len(message_divs)} блоков, всего собрано {len(all_posts)}")

        if stop or page_min_id is None:
            break
        before_id = page_min_id
        time.sleep(REQUEST_DELAY_SEC)

    return all_posts


def main():
    all_rows = []
    for channel in TARGET_CHANNELS:
        posts = collect_channel(channel)
        all_rows.extend(posts)

    # фильтр по окнам дат
    filtered = []
    for post in all_rows:
        if not post.datetime_utc:
            continue
        post_date = dtparser.isoparse(post.datetime_utc).date()
        if in_any_window(post_date):
            filtered.append(post)

    filtered.sort(key=lambda p: p.datetime_utc or "")

    with open(OUTPUT_CSV, "w", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(f, fieldnames=list(asdict(filtered[0]).keys()) if filtered else
                                 ["channel", "msg_id", "url", "datetime_utc", "text", "views",
                                  "reactions_total", "is_forwarded", "forwarded_from", "has_media",
                                  "links_in_text", "tags"])
        writer.writeheader()
        for post in filtered:
            writer.writerow(asdict(post))

    print(f"\nВсего постов собрано (все даты): {len(all_rows)}")
    print(f"Постов в окнах дат: {len(filtered)} -> сохранено в {OUTPUT_CSV}")

    # быстрая сводка по окнам
    for start, end in DATE_WINDOWS:
        window_posts = [
            p for p in filtered
            if start <= dtparser.isoparse(p.datetime_utc).date() <= end
        ]
        print(f"\n--- Окно {start} .. {end}: {len(window_posts)} постов ---")
        for p in window_posts:
            print(f"  [{p.datetime_utc}] tags={p.tags} | {p.text[:80]}")


if __name__ == "__main__":
    main()


=== Канал: postypashki_old ===
  страница 1: +14 блоков, всего собрано 12
  страница 2: +20 блоков, всего собрано 29
  страница 3: +18 блоков, всего собрано 45
  страница 4: +17 блоков, всего собрано 62

Всего постов собрано (все даты): 62
Постов в окнах дат: 12 -> сохранено в telegram_marketing_history.csv

--- Окно 2026-08-08 .. 2026-08-10: 5 постов ---
  [2026-08-08T06:05:37+00:00] tags=discount;sale_post;native_integration | Магистратура по e-commerce от РУДН и Wildberries — для тех, кто хочет развиватьс
  [2026-08-09T10:32:11+00:00] tags=discount;sale_post | Дайджест вакансий | Товарищи, собрали свежие стажировки и junior-вакансии по ана
  [2026-08-09T15:35:40+00:00] tags=other | Как найти стажировку зарубежом | Товарищи, пока идут последние |  6 часов финаль
  [2026-08-10T13:26:59+00:00] tags=discount;launch | Из процесс-менеджера в продуктового аналитика  | Друзья, видим ваши вопросы, мож
  [2026-08-10T17:33:06+00:00] tags=discount;sale_post | У России три пути: 18+, ***** и IT

In [4]:
import pandas as pd

df = pd.read_csv('telegram_marketing_history.csv')

df.head()

,channel,msg_id,url,datetime_utc,text,views,reactions_total,is_forwarded,forwarded_from,has_media,links_in_text,tags
0,postypashki_old,1834,https://t.me/postypashki_old/1834,2026-08-08T06:05:37+00:00,Магистратура по e-commerce от РУДН и Wildberri...,15.5K,NaN,False,NaN,True,https://university.rwb.ru/programs/e-commerce?...,discount;sale_post;native_integration
1,postypashki_old,1836,https://t.me/postypashki_old/1836,2026-08-09T10:32:11+00:00,"Дайджест вакансий | Товарищи, собрали свежие с...",18.3K,NaN,False,NaN,True,https://rabota-vtb.ru/career/135921123?utm_sou...,discount;sale_post
2,postypashki_old,1837,https://t.me/postypashki_old/1837,2026-08-09T15:35:40+00:00,"Как найти стажировку зарубежом | Товарищи, пок...",19.8K,NaN,False,NaN,True,https://t.me/m/p94YcePXNjcy | https://t.me/pos...,other
3,postypashki_old,1838,https://t.me/postypashki_old/1838,2026-08-10T13:26:59+00:00,Из процесс-менеджера в продуктового аналитика ...,16.5K,NaN,False,NaN,False,https://t.me/postypashki_old/1835 | https://ol...,discount;launch
4,postypashki_old,1840,https://t.me/postypashki_old/1840,2026-08-10T17:33:06+00:00,"У России три пути: 18+, ***** и IT | И кажется...",30.8K,NaN,False,NaN,False,https://t.me/postypashki_old/1757 | https://t....,discount;sale_post
